# GX13 — Autovalores: Brent vs aproximação de Beck (1992)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/gx13_autovalores.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*  
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## Descrição

O problema GX13 corresponde à placa plana com temperatura prescrita em $x=0$ (tipo 1) e convecção em $x=L$ (tipo 3). Os autovalores $\beta_m$ satisfazem a equação transcendental (Eq. 4.21):

$$\xi_m \cot(\xi_m) = -B, \qquad \xi_m = \beta_m L, \quad B = \frac{h L}{k}$$

Este notebook compara os autovalores obtidos pelo **método de Brent** com os valores **aproximados** pelas fórmulas de Beck (1992), com e sem a correção iterativa de Newton-Raphson (Eq. 4.26).

## Bibliotecas

In [1]:
import numpy as np
from scipy.optimize import brentq
import pandas as pd

## Parâmetro $B$

Altere o valor de `B` para explorar diferentes regimes de convecção. As fórmulas de Beck cobrem:

| Fórmula | Validade | Equação no livro |
|---|---|---|
| Eq. (4.22) | $m=1$, $-1 \leq B < -0{,}6$ | casos com $h < 0$ |
| Eq. (4.23) | $m=1,2,\ldots$, $-1 \leq B < 5$ | maioria dos problemas práticos |
| Eqs. (4.24)–(4.25) | $B > 5$ | convecção intensa |

Em todos os casos, a **Eq. (4.26)** aplica uma correção iterativa de Newton-Raphson para aumentar a precisão.

In [2]:
B = 2.0   # B = h·L/k  — altere aqui para explorar diferentes casos
M  = 5     # número de autovalores

print(f'B = {B:.4f}')

B = 2.0000


## Autovalores por Brent

A equação $\xi\cot(\xi) = -B$ é equivalente a:

$$f(\xi) = \frac{\xi}{\tan(\xi)} + B = 0$$

A função $\xi\cot(\xi)$ tem singularidades em $\xi = m\pi$ e vale zero em $\xi = (m-\tfrac{1}{2})\pi$. Para $B > 0$, cada raiz $\xi_m$ está no intervalo $\bigl((m-1)\pi + \varepsilon,\; m\pi - \varepsilon\bigr)$:

| $m$ | Intervalo de busca |
|---|---|
| 1 | $(\varepsilon,\; \pi - \varepsilon)$ |
| 2 | $(\pi + \varepsilon,\; 2\pi - \varepsilon)$ |
| 3 | $(2\pi + \varepsilon,\; 3\pi - \varepsilon)$ |

O método de Brent garante encontrar a raiz em cada intervalo.

In [3]:
eps = 1e-10

def f_gx13(xi, B):
    """Equação transcendental GX13: ξ·cot(ξ) + B = 0"""
    return xi / np.tan(xi) + B

xi_brent = np.array([
    brentq(f_gx13, (m-1)*np.pi + eps, m*np.pi - eps, args=(B,))
    for m in range(1, M+1)
])

print('Autovalores por Brent ξ_m = β_m · L:')
for m, xi in enumerate(xi_brent, 1):
    print(f'  ξ_{m} = {xi:.8f}')

Autovalores por Brent ξ_m = β_m · L:
  ξ_1 = 2.28892973
  ξ_2 = 5.08698509
  ξ_3 = 8.09616360
  ξ_4 = 11.17270587
  ξ_5 = 14.27635292


## Aproximações de Beck (1992)

### Eq. (4.22) — $m=1$, $-1 \leq B < -0{,}6$

$$\xi_1 = \left[3\left(1+B - \tfrac{1}{5}(1+B)^2\right)\right]^{1/2}$$

### Eq. (4.23) — $m=1,2,\ldots$, $-1 \leq B < 5$

$$\xi_m = \frac{\pi}{2}(2m-1)\left[1 + \frac{3}{2(B+3)}\left(\left(1 + \frac{16B(B+3)}{3(2m-1)^2\pi^2}\right)^{\!1/2} - 1\right)\right]$$

### Eqs. (4.24)–(4.25) — $B > 5$

$$A = \left[\left(\frac{3m\pi}{2B}\right)^2 + \left(1+\frac{1}{B}\right)^3\right]^{1/2}, \qquad
\xi_m = m\pi - \left(A + \frac{3m\pi}{2B}\right)^{1/3} + \left(A - \frac{3m\pi}{2B}\right)^{1/3}$$

### Eq. (4.26) — Correção de Newton-Raphson (aplicada iterativamente)

Partindo do valor aproximado $\xi'_m$, aplica-se repetidamente:

$$\xi_m \leftarrow \xi_m - \frac{B\tan\xi_m + \xi_m}{B/\cos^2\!\xi_m + 1}$$

até que a correção seja inferior a um critério de convergência $\psi$. Cada iteração resolve o sistema $F(\xi) = B\tan\xi + \xi = 0$, que é equivalente à Eq. (4.21).

In [4]:
def beck_xi(m, B):
    """Aproximação inicial de Beck para ξ_m de GX13."""
    if m == 1 and -1.0 <= B < -0.6:          # Eq. (4.22)
        return np.sqrt(3 * (1 + B - (1 + B)**2 / 5))
    elif B <= 5:                               # Eq. (4.23)
        n = 2*m - 1
        inner = (1 + 16*B*(B+3) / (3*n**2*np.pi**2))**0.5 - 1
        return (np.pi/2) * n * (1 + 3/(2*(B+3)) * inner)
    else:                                       # Eqs. (4.24)–(4.25)
        A = np.sqrt((3*m*np.pi / (2*B))**2 + (1 + 1/B)**3)
        t1 = (A + 3*m*np.pi / (2*B))**(1/3)
        t2 = (A - 3*m*np.pi / (2*B))**(1/3)
        return m*np.pi - t1 + t2

def newton_iter(xi0, B, tol=1e-12, nmax=100):
    """Newton-Raphson iterativo — Eq. (4.26)."""
    xi = xi0
    for _ in range(nmax):
        F  = B * np.tan(xi) + xi
        Fp = B / np.cos(xi)**2 + 1
        dxi = F / Fp
        xi -= dxi
        if abs(dxi) < tol:
            return xi
    return xi   # retorna o melhor valor encontrado

xi_beck    = np.array([beck_xi(m, B)                     for m in range(1, M+1)])
xi_beck_nr = np.array([newton_iter(beck_xi(m, B), B)    for m in range(1, M+1)])

## Tabela de comparação

A tabela abaixo mostra os autovalores obtidos por Brent $\xi_m^{\text{Brent}}$, a aproximação inicial de Beck $\xi_m^{\text{Beck}}$, a aproximação com NR iterativo $\xi_m^{\text{Beck+NR}}$ e os erros relativos percentuais.

In [5]:
erro_beck    = 100 * np.abs(xi_beck    - xi_brent) / xi_brent
erro_beck_nr = 100 * np.abs(xi_beck_nr - xi_brent) / xi_brent

df = pd.DataFrame({
    'm'               : range(1, M+1),
    'ξ_Brent'         : xi_brent,
    'ξ_Beck'          : xi_beck,
    'Erro Beck (%)'   : erro_beck,
    'ξ_Beck+NR'       : xi_beck_nr,
    'Erro+NR (%)'     : erro_beck_nr,
})
df = df.set_index('m')

pd.set_option('display.float_format', '{:.6f}'.format)
print(f'B = {B:.4f}\n')
print(df.to_string())

B = 2.0000

    ξ_Brent    ξ_Beck  Erro Beck (%)  ξ_Beck+NR  Erro+NR (%)
m                                                           
1  2.288930  2.292062       0.136825   2.288930     0.000000
2  5.086985  5.087134       0.002925   5.086985     0.000000
3  8.096164  8.096181       0.000219   8.096164     0.000000
4 11.172706 11.172710       0.000034  11.172706     0.000000
5 14.276353 14.276354       0.000008  14.276353     0.000000


## Observações

- Para **$B \geq 1$**: a aproximação de Beck (Eq. 4.23) fornece um valor inicial razoável, e o NR iterativo converge à precisão de máquina em poucas iterações.
- Para **$B > 5$**: as Eqs. (4.24)–(4.25) são muito precisas já como aproximação inicial (erro < 0,01%), e o NR converge imediatamente.
- Para **$B < 1$**: a aproximação inicial de Beck pode ser insuficiente para que o NR convirja. Nesse caso, recomenda-se usar diretamente o método de Brent (`brentq`) como referência.